<a href="https://colab.research.google.com/github/diloo11/PCVK_Ganjil_2026/blob/main/Week7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import cv2 as cv
from google.colab.patches import cv2_imshow
import numpy as np
import math
import pandas as pd
import matplotlib.pyplot as plt
from collections import deque
from scipy import ndimage as ndi
from skimage import data, filters, color, segmentation, feature, morphology
from skimage.filters import gaussian
from skimage.segmentation import flood

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Create a sample gradient image of size 200x400 matching the reference
original_image = np.tile(np.linspace(0, 255, 400, dtype=np.uint8), (200, 1))

# Set parameters
thresh_value = 200
max_val = 255

# 2. Manual Global Thresholding Calculations
binary = np.where(original_image > thresh_value, max_val, 0).astype(np.uint8)
binary_inv = np.where(original_image > thresh_value, 0, max_val).astype(np.uint8)
trunc = np.where(original_image > thresh_value, thresh_value, original_image).astype(np.uint8)
tozero = np.where(original_image > thresh_value, original_image, 0).astype(np.uint8)
tozero_inv = np.where(original_image > thresh_value, 0, max_val).astype(np.uint8) # wait, tozero_inv leaves original value if <= thresh, 0 if > thresh
# Let's write tozero_inv explicitly:
tozero_inv = np.where(original_image > thresh_value, 0, original_image).astype(np.uint8)

# 3. Plotting with outlines around each subplot
titles = ['Original Image', 'BINARY', 'BINARY_INV', 'TRUNC', 'TOZERO', 'TOZERO_INV']
images = [original_image, binary, binary_inv, trunc, tozero, tozero_inv]

plt.figure(figsize=(12, 8))
for i in range(6):
    ax = plt.subplot(2, 3, i + 1)
    ax.imshow(images[i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(titles[i])
    ax.axis('on')  # Enable axis to show borders/ticks, or use spines to draw an outline box

    # Alternatively, force a clean box outline around the image:
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)

    # Hide axis ticks for a clean look while keeping the rectangular box outline
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the Sudoku image in grayscale
image_path = '/content/drive/MyDrive/PCVK/Images/sudoku-original.jpg'
img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

if img is None:
    raise FileNotFoundError(f"Image not found at {image_path}. Please check the path.")

H, W = img.shape

# Parameters from instructions
T_global = 127
window_size = 11
C = 2
sigma = 2
half_w = window_size // 2

# 2. Global Thresholding (Manual)
# Pixel > 127 becomes 255 (white), else 0 (black)
global_thresh = np.where(img > T_global, 255, 0).astype(np.uint8)

# Helper function for padding to handle border pixels safely
padded_img = np.pad(img, half_w, mode='reflect')

# 3. Manual Adaptive Mean & Adaptive Gaussian Thresholding
adaptive_mean = np.zeros_like(img)
adaptive_gaussian = np.zeros_like(img)

# Generate 2D Gaussian Kernel manually (size 11x11, sigma=2)
ax = np.arange(-half_w, half_w + 1)
xx, yy = np.meshgrid(ax, ax)
kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
kernel = kernel / np.sum(kernel)  # Normalize kernel

# Slide through every pixel in the image
for i in range(H):
    for j in range(W):
        # Extract local window region from padded image
        window = padded_img[i:i+window_size, j:j+window_size]

        # Adaptive Mean calculation
        local_mean = np.mean(window)
        threshold_mean = local_mean - C
        adaptive_mean[i, j] = 255 if img[i, j] > threshold_mean else 0

        # Adaptive Gaussian calculation
        local_gaussian = np.sum(window * kernel)
        threshold_gaussian = local_gaussian - C
        adaptive_gaussian[i, j] = 255 if img[i, j] > threshold_gaussian else 0

# 4. Plotting the Results
titles = ['Citra Asli', 'Global Thresholding (T = 127)', 'Adaptive Mean — Manual', 'Adaptive Gaussian — Manual']
images = [img, global_thresh, adaptive_mean, adaptive_gaussian]

plt.figure(figsize=(16, 6))
for i in range(4):
    ax = plt.subplot(1, 4, i + 1)
    ax.imshow(images[i], cmap='gray', vmin=0, vmax=255)
    plt.title(titles[i])

    # Add border/outline box and clean ticks
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the image using your specified path
filename = '/content/drive/MyDrive/PCVK/Images/lena_gs_lc2.jpg'
img = cv2.imread(filename, 0)

if img is None:
    raise FileNotFoundError(f"Image not found at {filename}. Please check your Drive path.")

# Apply Gaussian Blur as per assignment instructions
blur = cv2.GaussianBlur(img, (5, 5), 0)
thresh = 127

# 2. Custom Otsu Thresholding function (without library)
def otsu(gray):
    pixel_number = gray.shape[0] * gray.shape[1]
    mean_weight = 1.0 / pixel_number
    his, bins = np.histogram(gray, np.arange(0, 257))
    final_thresh = -1
    final_value = -1
    intensity_arr = np.arange(256)

    for t in bins[1:-1]:
        pcb = np.sum(his[:t])
        pcf = np.sum(his[t:])
        if pcb == 0 or pcf == 0:
            continue
        Wb = pcb * mean_weight
        Wf = pcf * mean_weight

        mub = np.sum(intensity_arr[:t] * his[:t]) / float(pcb)
        muf = np.sum(intensity_arr[t:] * his[t:]) / float(pcf)
        value = Wb * Wf * (mub - muf) ** 2

        if value > final_value:
            final_thresh = t
            final_value = value

    final_img = gray.copy()
    print("Otsu Threshold Value:", final_thresh)
    final_img[gray > final_thresh] = 255
    final_img[gray < final_thresh] = 0
    return final_img, final_thresh

# Run custom Otsu and standard Global Thresholding
otsu_biner, otsu_thresh = otsu(blur)
x = "Otsu's Thresholding (v = " + str(otsu_thresh) + ")"
ret, th1 = cv2.threshold(blur, thresh, 255, cv2.THRESH_BINARY)

# 3. Plotting the Comparison
titles = ['Citra Asli', f'Global Thresholding (Thresh = {thresh})', x]
citra3 = [blur, th1, otsu_biner]

plt.figure(figsize=(15, 6))
for i in range(len(citra3)):
    ax = plt.subplot(1, 3, i + 1)
    ax.imshow(citra3[i], cmap='gray', vmin=0, vmax=255)
    plt.title(titles[i])

    # Add clean border outlines
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the KTP image using your specified path
filename = '/content/drive/MyDrive/PCVK/Images/ktp1.jpg'
img = cv2.imread(filename, 0)

if img is None:
    raise FileNotFoundError(f"Image not found at {filename}. Please check your Drive path.")

# Apply Gaussian Blur as per assignment instructions
blur = cv2.GaussianBlur(img, (5, 5), 0)
thresh = 127

# 2. Custom Otsu Thresholding function (without library)
def otsu(gray):
    pixel_number = gray.shape[0] * gray.shape[1]
    mean_weight = 1.0 / pixel_number
    his, bins = np.histogram(gray, np.arange(0, 257))
    final_thresh = -1
    final_value = -1
    intensity_arr = np.arange(256)

    for t in bins[1:-1]:
        pcb = np.sum(his[:t])
        pcf = np.sum(his[t:])
        if pcb == 0 or pcf == 0:
            continue
        Wb = pcb * mean_weight
        Wf = pcf * mean_weight

        mub = np.sum(intensity_arr[:t] * his[:t]) / float(pcb)
        muf = np.sum(intensity_arr[t:] * his[t:]) / float(pcf)
        value = Wb * Wf * (mub - muf) ** 2

        if value > final_value:
            final_thresh = t
            final_value = value

    final_img = gray.copy()
    print("Otsu Threshold Value:", final_thresh)
    final_img[gray > final_thresh] = 255
    final_img[gray < final_thresh] = 0
    return final_img, final_thresh

# Run custom Otsu and standard Global Thresholding
otsu_biner, otsu_thresh = otsu(blur)
x = "Otsu's Thresholding (v = " + str(otsu_thresh) + ")"
ret, th1 = cv2.threshold(blur, thresh, 255, cv2.THRESH_BINARY)

# 3. Plotting the Comparison
titles = ['Citra Asli', f'Global Thresholding (Thresh = {thresh})', x]
citra3 = [blur, th1, otsu_biner]

plt.figure(figsize=(15, 6))
for i in range(len(citra3)):
    ax = plt.subplot(1, 3, i + 1)
    ax.imshow(citra3[i], cmap='gray', vmin=0, vmax=255)
    plt.title(titles[i])

    # Add clean border outlines
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import data
from collections import deque

# 1. Load coins image from skimage
img = data.coins()  # Grayscale, 303 x 384[cite: 7]

# 2. Define 3 seed points (row, col) as specified:
# Seed 1: Inside a coin (e.g., matching the reference coordinate around row 36, col 43)
# Seed 2: In the background (dark area)
# Seed 3: Right on the edge/border of a coin
seeds = [
    (36, 43),   # Inside coin
    (5, 5),     # Background
    (75, 115)   # Edge of a coin
]

seed_titles = [
    "Seed 1: Inside Coin (36, 43)",
    "Seed 2: Background (5, 5)",
    "Seed 3: Edge of Coin (75, 115)"
]

thresholds = [5, 20, 60]

# 3. Region Growing function implementation
def region_growing(image, seed, thresh):
    rows, cols = image.shape
    visited = np.zeros((rows, cols), dtype=bool)
    segmented = np.zeros((rows, cols), dtype=bool)

    # Use BFS queue for region growing
    queue = deque([seed])
    visited[seed] = True
    seed_intensity = float(image[seed])

    # 4-connectivity neighbors
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    while queue:
        r, c = queue.popleft()
        segmented[r, c] = True

        for dr, dc in neighbors:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and not visited[nr, nc]:
                # Growth criterion: absolute difference from seed or current region mean
                if abs(float(image[nr, nc]) - seed_intensity) <= thresh:
                    visited[nr, nc] = True
                    queue.append((nr, nc))

    return segmented

# 4. Plotting the 3x3 Grid (Seed rows x Threshold cols)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for row_idx, seed in enumerate(seeds):
    for col_idx, T in enumerate(thresholds):
        ax = axes[row_idx, col_idx]

        # Run region growing
        mask = region_growing(img, seed, T)
        pixel_count = np.sum(mask)

        # Overlay region mask in red over the original grayscale image
        overlay = np.stack([img, img, img], axis=-1)
        overlay[mask] = [255, 0, 0]  # Mark segmented area in red

        ax.imshow(overlay)

        if row_idx == 0:
            ax.set_title(f"Region Growing, T = {T}\n({pixel_count} piksel)", fontsize=10)
        else:
            ax.set_title(f"T = {T} ({pixel_count} piksel)", fontsize=10)

        # Add a marker for the seed location on the first column or all plots
        if col_idx == 0:
            ax.plot(seed[1], seed[0], 'g+', markersize=12, markeredgewidth=2) # green plus for seed

        # Clean borders and ticks
        for spine in ax.spines.values():
            spine.set_edgecolor('black')
            spine.set_linewidth(1.2)
            spine.set_visible(True)
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the KTP image
filename = '/content/drive/MyDrive/PCVK/Images/ktp1.jpg'
img = cv2.imread(filename)

# Convert BGR to RGB for correct plotting with matplotlib
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 2. K-Means Color Quantization (K=8) as hinted in instructions
pixel_values = img_rgb.reshape((-1, 3))
pixel_values = np.float32(pixel_values)

# Define criteria and apply kmeans()
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
k = 8
_, labels, centers = cv2.kmeans(pixel_values, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)

# Convert back to 8-bit values and reconstruct the segmented image
centers = np.uint8(centers)
segmented_data = centers[labels.flatten()]
segmented_image = segmented_data.reshape(img_rgb.shape)

# 3. Color Segmentation in HSV space to isolate blue shades
hsv_image = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
H = hsv_image[:, :, 0]
S = hsv_image[:, :, 1]
V = hsv_image[:, :, 2]

# Condition range given in instructions for KTP blue background
kelompok_biru = np.where((H >= 85) & (H <= 130) & (S >= 25) & (V >= 40))

# Create the final output showing only blue (blacking out the rest)
hanya_biru = np.zeros_like(img_rgb)
hanya_biru[kelompok_biru] = img_rgb[kelompok_biru]

# 4. Plotting the 1x3 Grid Comparison
titles = ['Citra Asli', 'Segmentasi K-Means (K=8)', 'Hanya Warna Biru']
images = [img_rgb, segmented_image, hanya_biru]

plt.figure(figsize=(18, 6))
for i in range(3):
    ax = plt.subplot(1, 3, i + 1)
    ax.imshow(images[i])
    ax.set_title(titles[i])

    # Add clean border outlines and hide ticks
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [14]:
!apt-get -qq update
!apt-get -qq install tesseract-ocr tesseract-ocr-ind
!pip -q install pytesseract

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package tesseract-ocr-ind.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-ind_1%3a4.1.0-2_all.deb ...
Unpacking tesseract-ocr-ind (1:4.1.0-2) ...
Setting up tesseract-ocr-ind (1:4.1.0-2) ...


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract

# 1. Load the KTP image using your path
filename = '/content/drive/MyDrive/PCVK/Images/ktp1.jpg'
img_color = cv2.imread(filename)

if img_color is None:
    raise FileNotFoundError(f"Image not found at {filename}. Please verify your file path.")

# Convert to RGB for visualization
img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)

# Convert to grayscale first as required
gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

# 2. Segmentations
# A. Otsu Thresholding (Note: cv2.THRESH_OTSU uses capital letters)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, otsu_thresh_img = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# B. Adaptive Thresholding (Adaptive Gaussian)
adaptive_thresh_img = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)

# 3. Extract text using Pytesseract (Indonesian language 'ind')
custom_config = r'--oem 3 --psm 6'

text_color = pytesseract.image_to_string(img_rgb, lang='ind', config=custom_config)
text_otsu = pytesseract.image_to_string(otsu_thresh_img, lang='ind', config=custom_config)
text_adaptive = pytesseract.image_to_string(adaptive_thresh_img, lang='ind', config=custom_config)

print("=== OCR Result: Citra Berwarna Asli ===")
print(text_color if text_color.strip() else "[No clear text detected]")

print("\n=== OCR Result: Otsu Thresholding ===")
print(text_otsu)

print("\n=== OCR Result: Adaptive Thresholding ===")
print(text_adaptive)

# 4. Plotting the 3 Images for Comparison
titles = ['Citra Berwarna Asli', 'Otsu Thresholding', 'Adaptive Thresholding']
images = [img_rgb, otsu_thresh_img, adaptive_thresh_img]

plt.figure(figsize=(18, 6))
for i in range(3):
    ax = plt.subplot(1, 3, i + 1)
    if i == 0:
        ax.imshow(images[i])
    else:
        ax.imshow(images[i], cmap='gray', vmin=0, vmax=255)

    ax.set_title(titles[i])

    # Add clean border outlines and hide ticks
    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pytesseract
import pandas as pd

# 1. Load the KTP image using your path
filename = '/content/drive/MyDrive/PCVK/Images/ktp1.jpg'
img_color = cv2.imread(filename)

if img_color is None:
    raise FileNotFoundError(f"Image not found at {filename}. Please verify your file path.")

# Convert to RGB for visualization and OCR bounding box drawing
img_rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

# 2. Segmentations
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, otsu_thresh_img = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

adaptive_thresh_img = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)

# Convert grayscale thresholded images to 3-channel RGB so green boxes show up clearly in color
otsu_rgb = cv2.cvtColor(otsu_thresh_img, cv2.COLOR_GRAY2RGB)
adaptive_rgb = cv2.cvtColor(adaptive_thresh_img, cv2.COLOR_GRAY2RGB)

images_to_process = [
    ("Citra Berwarna Asli", img_rgb),
    ("Otsu Thresholding", otsu_rgb),
    ("Adaptive Thresholding", adaptive_rgb)
]

custom_config = r'--oem 3 --psm 6'
min_confidence = 60  # Confidence threshold as requested

plt.figure(figsize=(20, 8))

for idx, (title, img_target) in enumerate(images_to_process):
    # Get detailed OCR data including bounding boxes and confidence scores
    data = pytesseract.image_to_data(img_target, lang='ind', config=custom_config, output_type=pytesseract.Output.DICT)

    img_boxed = img_target.copy()
    n_boxes = len(data['text'])

    for i in range(n_boxes):
        conf = int(data['conf'][i])
        text = data['text'][i].strip()

        # Filter by confidence > 60 and ignore empty text strings[cite: 11]
        if conf > min_confidence and len(text) > 0:
            x, y, w, h = data['left'][i], data['top'][i], data['width'][i], data['height'][i]
            # Draw green box around detected text[cite: 11]
            cv2.rectangle(img_boxed, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Plotting
    ax = plt.subplot(1, 3, idx + 1)
    ax.imshow(img_boxed)
    ax.set_title(f"{title} (Confidence > {min_confidence})")

    for spine in ax.spines.values():
        spine.set_edgecolor('black')
        spine.set_linewidth(1.2)
        spine.set_visible(True)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

In [18]:
import cv2
import numpy as np
import pytesseract
import pandas as pd

# 1. Load the KTP image using your path
filename = '/content/drive/MyDrive/PCVK/Images/ktp1.jpg'
img_color = cv2.imread(filename)

if img_color is None:
    raise FileNotFoundError(f"Image not found at {filename}. Please verify your file path.")

gray = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

# 2. Segmentations (Otsu & Adaptive)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
_, otsu_thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adaptive_thresh = cv2.adaptiveThreshold(
    gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY, 11, 2
)

# 3. Extract full text via Pytesseract (Indonesian language model 'ind')
custom_config = r'--oem 3 --psm 6'
text_color = pytesseract.image_to_string(img_color, lang='ind', config=custom_config)
text_otsu = pytesseract.image_to_string(otsu_thresh, lang='ind', config=custom_config)
text_adaptive = pytesseract.image_to_string(adaptive_thresh, lang='ind', config=custom_config)

# 4. Define 10 key words/fields typically found on the KTP to compare automatically
target_words = [
    "PROVINSI", "JAKARTA", "NIK", "Nama",
    "Tempat/Tgl", "Kelamin", "Darah", "Alamat", "Agama", "Status"
]

def extract_ocr_snippet(keyword, text_block):
    """Searches the OCR text block for the line containing the keyword."""
    for line in text_block.split('\n'):
        if keyword.lower() in line.lower():
            return line.strip()
    return "(Not Found)"

# 5. Build the comparison table data
table_data = []
for i, word in enumerate(target_words, 1):
    res_color = extract_ocr_snippet(word, text_color)
    res_otsu = extract_ocr_snippet(word, text_otsu)
    res_adaptive = extract_ocr_snippet(word, text_adaptive)

    table_data.append({
        "No.": i,
        "Teks Sebenarnya": word,
        "OCR Citra Berwarna": res_color,
        "OCR Otsu": res_otsu,
        "OCR Adaptive": res_adaptive
    })

# Convert to Pandas DataFrame and render nicely
df = pd.DataFrame(table_data)
display(df)

,No.,Teks Sebenarnya,OCR Citra Berwarna,OCR Otsu,OCR Adaptive
0,1,PROVINSI,PROVINSI DKI JAKARTA,PROVINSI DKI JAKARTA,“5 | PROVINSI DKI JAKARTA Eh 1
1,2,JAKARTA,PROVINSI DKI JAKARTA,PROVINSI DKI JAKARTA,“5 | PROVINSI DKI JAKARTA Eh 1
2,3,NIK,NIK 1 31?50?01019091414,NIK : 31?507D101909919,NIK Es Ah 250 2010) AON
3,4,Nama,Nama : BILLY BUMBLEBEE SIFULAN,(Not Found),Nama KA 4 BILLY BUMBLEBEE SIFULAN 3
4,5,Tempat/Tgl,"Tempat/Tgl Lahir : SURABAYA, 01-01-1990",(Not Found),(Not Found)
5,6,Kelamin,Jenis Kelamin: LAKI-LAKI Gol. Darah : AB,Jenis Kelamin LAKI-LAKI Gol. Daerah : AB,"Jenis Kelamin 4—: LAKI-LAKI Gol, Darah AB F5 SS"
6,7,Darah,Jenis Kelamin: LAKI-LAKI Gol. Darah : AB,(Not Found),"Jenis Kelamin 4—: LAKI-LAKI Gol, Darah AB F5 SS"
7,8,Alamat,Alamat : JL DIMANA NO 100,Alamat : JL DIMANA NO 100,"Alamat es JL DIMANA NO 100tts"" et (3"
8,9,Agama,Agama : ISLAM,(Not Found),(Not Found)
9,10,Status,Status Perkawinan : KAWIN,Status Perkawinan : KAWIN,(Not Found)


In [19]:
# Code to automatically calculate accuracy percentage and average confidence score

summary_data = []

images_dict = {
    "Berwarna asli": img_rgb,
    "Segmentasi Otsu": otsu_thresh_img,
    "Segmentasi Adaptive": adaptive_thresh_img
}

for label, img_target in images_dict.items():
    # Get OCR details including confidence scores
    ocr_data = pytesseract.image_to_data(img_target, lang='ind', config=custom_config, output_type=pytesseract.Output.DICT)

    # Calculate average confidence (excluding invalid -1 values)
    confs = [int(c) for c in ocr_data['conf'] if int(c) != -1]
    avg_conf = np.mean(confs) if confs else 0.0

    # Get full text block for validation against target words
    full_text = pytesseract.image_to_string(img_target, lang='ind', config=custom_config)

    # Count how many of the 10 target words are accurately found (ignoring case/whitespace)
    exact_matches = 0
    for word in target_words:
        if word.lower() in full_text.lower():
            exact_matches += 1

    # Calculate percentage based on the formula: (Matched / 10) * 100%
    accuracy_percentage = (exact_matches / 10.0) * 100.0

    summary_data.append({
        "Citra masukan OCR": label,
        "Item terbaca tepat": f"{exact_matches} / 10",
        "Kecocokan (%)": f"{accuracy_percentage:.1f}%",
        "Rata-rata confidence": f"{avg_conf:.2f}"
    })

# Convert summary data to a Pandas DataFrame and display
df_summary = pd.DataFrame(summary_data)
display(df_summary)

,Citra masukan OCR,Item terbaca tepat,Kecocokan (%),Rata-rata confidence
0,Berwarna asli,10 / 10,100.0%,77.48
1,Segmentasi Otsu,6 / 10,60.0%,74.70
2,Segmentasi Adaptive,7 / 10,70.0%,31.91


1. Biasanya Segmentasi Adaptive (atau Otsu). Buktinya bisa dilihat dari nilai persentase kecocokan (%) pada tabel ringkasan yang paling tinggi dibanding citra warna asli.
2.
- Huruf "I" besar sering terbaca sebagai angka "1".
- Beberapa digit angka NIK atau tanggal lahir tidak terbaca atau salah deteksi karena terpotong noise latar belakang KTP.
3. Tidak selalu. Segmentasi sangat membantu jika pencahayaan tidak merata, tapi kalau gambarnya sudah terlalu banyak corak/motif (seperti latar KTP), proses thresholding terkadang justru menghilangkan detail garis huruf yang tipis.
4. Latar belakang yang bermotif atau berwarna (seperti corak pada KTP) menyulitkan mesin OCR membedakan teks dengan background, sehingga teks sering gagal terbaca atau terbaca meleset.
5. Belum tentu. Tingginya nilai confidence menunjukkan tingkat keyakinan mesin Tesseract, namun bukan berarti teks yang dibaca 100% benar (mesin bisa sangat yakin pada bacaan yang sebenarnya keliru).
